# Estimando distribuciones (parte 1)
El objetivo de esta notebook es explorar una primera manera de aproximar $p(y|x)$ y $p(x|y)$ en un set de datos tabular. En este set de datos $x$ tiene valores discretos, $x\in\mathbb{D}^k$, y el target $y$ es un booleano, $y\in\{0,1\}$.

## Imports

In [1]:
import numpy as np
import pandas as pd
import random

## Cargamos los datos

In [2]:
df = pd.read_csv('./tennis.csv', delimiter=',', header=0)
df

,Day,Outlook,Temp,Humidity,Wind,Tennis
0,D1,Sunny,Hot,High,Weak,No
1,D2,Sunny,Hot,High,Strong,No
2,D3,Overcast,Hot,High,Weak,Yes
3,D4,Rain,Mild,High,Weak,Yes
4,D5,Rain,Cool,Normal,Weak,Yes
5,D6,Rain,Cool,Normal,Strong,No
6,D7,Overcast,Cool,Normal,Strong,Yes
7,D8,Sunny,Mild,High,Weak,No
8,D9,Sunny,Cool,Normal,Weak,Yes
9,D10,Rain,Mild,Normal,Weak,Yes


### Eliminamos la columna Day 

In [3]:
df = df.drop('Day', axis=1)
df

,Outlook,Temp,Humidity,Wind,Tennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [4]:
X_names = df.columns.to_list()[:-1]
X_names

['Outlook', 'Temp', 'Humidity', 'Wind']

Guardamos en la variable $X$ todas las features del dataset.

In [5]:
X = df.iloc[:,0:-1]
X

,Outlook,Temp,Humidity,Wind
0,Sunny,Hot,High,Weak
1,Sunny,Hot,High,Strong
2,Overcast,Hot,High,Weak
3,Rain,Mild,High,Weak
4,Rain,Cool,Normal,Weak
5,Rain,Cool,Normal,Strong
6,Overcast,Cool,Normal,Strong
7,Sunny,Mild,High,Weak
8,Sunny,Cool,Normal,Weak
9,Rain,Mild,Normal,Weak


In [6]:
Y_name = df.columns.to_list()[-1]
Y_name

'Tennis'

Guardamos en $Y$ el objetivo

In [7]:
Y = df.iloc[:,-1]
Y

0      No
1      No
2     Yes
3     Yes
4     Yes
5      No
6     Yes
7      No
8     Yes
9     Yes
10    Yes
11    Yes
12    Yes
13     No
Name: Tennis, dtype: str

## Construimos una tabla de observaciones

En este paso vamos a crear una tabla de observaciones. Esta tabla tiene que contener la frecuencia de cada observación. Para este ejemplo tomaremos a $x$ como **Outlook**.

Calcule las dimensiones de la tabla

In [8]:
# Cantidad total de elementos
N = X['Outlook'].count()

# Elementos únicos de la clase Outlook
xvalues = X['Outlook'].unique()
dimx = len(xvalues)

# Elementos únicos del objetivo
yvalues = Y.unique()
dimy = len(yvalues)

print(f'Cantidad total de elementos: {N}')
print(f'Elementos únicos de la clase Outlook: {xvalues}')
print(f'Elementos únicos del objetivo: {yvalues}')

Cantidad total de elementos: 14
Elementos únicos de la clase Outlook: <StringArray>
['Sunny', 'Overcast', 'Rain']
Length: 3, dtype: str
Elementos únicos del objetivo: <StringArray>
['No', 'Yes']
Length: 2, dtype: str


Calculamos la tabla de frecuencia.

In [9]:
obs = pd.DataFrame(0, columns=yvalues, index=xvalues)

## Llene la tabla de observaciones

obs.loc['Sunny', 'No'] = len(df[(df['Outlook'] == 'Sunny') & (df['Tennis'] == 'No')])
obs.loc['Sunny', 'Yes'] = len(df[(df['Outlook'] == 'Sunny') & (df['Tennis'] == 'Yes')])

obs.loc['Overcast', 'No'] = len(df[(df['Outlook'] == 'Overcast') & (df['Tennis'] == 'No')])
obs.loc['Overcast', 'Yes'] = len(df[(df['Outlook'] == 'Overcast') & (df['Tennis'] == 'Yes')])

obs.loc['Rain', 'No'] = len(df[(df['Outlook'] == 'Rain') & (df['Tennis'] == 'No')])
obs.loc['Rain', 'Yes'] = len(df[(df['Outlook'] == 'Rain') & (df['Tennis'] == 'Yes')])

obs

,No,Yes
Sunny,3,2
Overcast,0,4
Rain,2,3


## Aproximación de la distribución conjunta $p(x,y)$

Tome a $x$ como Outlook y aproxime la distribución conjunta utilizando la tabla de observaciones. 

In [10]:
joint_x_y = obs / N
joint_x_y

,No,Yes
Sunny,0.214286,0.142857
Overcast,0.000000,0.285714
Rain,0.142857,0.214286


1. ¿Qué significa el valor calculado en los índices "Sunny", "Yes"?

El valor obtenido de la probablidad conjunta para el caso "Sunny"-"Yes" determina distribución de días de mi dataset en que se cumplen ambas condiciones al mismo tiempo.

2. ¿Justifique el resultado de la pareja "Overcast", "No"?

En este caso, para ambas casuisticas al mimso tiempo no tenemos observaciones en nuestro sample de datos que presenten estas condiciones.

## Aproximamos $p(y|x)$

Tome a $x$ como **Outlook** y estime la probabilidad condicional de $y$ dado $x$. Luego realice una muestra de 10 valores de $y$ dado $x = Sunny$.

Calculamos la cantidad de entradas por cada valor distinto de $x$.

In [11]:
m = obs.sum(axis=1)
obs['m'] = m
m

Sunny       5
Overcast    4
Rain        5
dtype: int64

Calculamos la cantidad de entradas por cada valor distinto de $y$.

In [12]:
l = obs.sum(axis=0)
obs.loc['l'] = l
l

No      5
Yes     9
m      14
dtype: int64

In [13]:
obs

,No,Yes,m
Sunny,3,2,5
Overcast,0,4,4
Rain,2,3,5
l,5,9,14


Calcule la probabilidad condicional de $y$ dado $x$.

In [14]:
p_y_x = pd.DataFrame(0.0, columns=yvalues, index=xvalues)

## Llene la tabla de probabilidades condicionales p(y|x)

p_y_x.loc['Sunny', 'No'] = obs.loc['Sunny', 'No'] /  obs.loc['Sunny','m']
p_y_x.loc['Sunny', 'Yes'] = obs.loc['Sunny', 'Yes'] /  obs.loc['Sunny','m']
p_y_x.loc['Overcast', 'No'] = obs.loc['Overcast', 'No'] /  obs.loc['Overcast','m']
p_y_x.loc['Overcast', 'Yes'] = obs.loc['Overcast', 'Yes'] /  obs.loc['Overcast','m']
p_y_x.loc['Rain', 'No'] = obs.loc['Rain', 'No'] /  obs.loc['Rain','m']
p_y_x.loc['Rain', 'Yes'] = obs.loc['Rain', 'Yes'] /  obs.loc['Rain','m']

p_y_x

,No,Yes
Sunny,0.6,0.4
Overcast,0.0,1.0
Rain,0.4,0.6


3. ¿La suma de cada fila siempre tiene que dar 1? ¿Por qué?

Sí. Cada fila representa $p(y|x=x_i)$ para un valor fijo de $x_i$, y $y$ es una variable que sólo puede tomar los valores "No" o "Yes".

4. ¿Y si la suma de las columnas?

No tiene por qué dar 1. Sumar una columna equivale a $\sum p(y=y_j|x)$, es decir, sumar la probabilidad condicional de un mismo $y_j$ a través de distintos valores de $x$, que no están pesados por $p(x)$ ni forman una distribución de probabilidad entre sí. De hecho en la tabla obtenida, la columna "No" suma 0.6+0.0+0.4=1.0 y la columna "Yes" suma 0.4+1.0+0.6=2.0.

Realice 10 muestras de $y$ dado $x = Sunny$.

Puede utilizar la función random.choice de numpy. https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html

In [15]:
sampled_values = np.random.choice(yvalues, size=10, p=p_y_x.loc['Sunny'].values)

sampled_values

array(['Yes', 'Yes', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes'],
      dtype=object)

5. ¿Qué pasaría si utilizamos $x = Overcast$ en vez de $x = Sunny$? ¿Tiene sentido que pase esto? ¿Por qué?

In [16]:
sampled_values_ov = np.random.choice(yvalues, size=10, p=p_y_x.loc['Overcast'].values)

sampled_values_ov

array(['Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes',
       'Yes'], dtype=object)

Tiene sentido que esto pase, ya que para la muestra de 14 datos que tenemos siempre tenemos Y = "Yes", entonces al hacer una muestra random basado en ese muestreo obtendremos un comportamiento similar a esa foto que tenemos de los datos. 

## Aproximamos $p(x|y)$
Tome a $x$ como Outlook y estime la probabilidad condicional de $x$ dado $y$ basandose en la tabla de observaciones. Luego realice 10 muestras de $x$ dado $y = Yes$.

$p(x|y)$

In [17]:
p_x_y = pd.DataFrame(0.0, columns=yvalues, index=xvalues)

## Llene la tabla de probabilidades condicionales p(x|y)
p_x_y.loc['Sunny', 'No'] = obs.loc['Sunny', 'No'] / obs.loc['l', 'No']
p_x_y.loc['Sunny', 'Yes'] = obs.loc['Sunny', 'Yes'] / obs.loc['l', 'Yes']
p_x_y.loc['Overcast', 'No'] = obs.loc['Overcast', 'No'] / obs.loc['l', 'No']
p_x_y.loc['Overcast', 'Yes'] = obs.loc['Overcast', 'Yes'] / obs.loc['l', 'Yes']
p_x_y.loc['Rain', 'No'] = obs.loc['Rain', 'No'] / obs.loc['l', 'No']
p_x_y.loc['Rain', 'Yes'] = obs.loc['Rain', 'Yes'] / obs.loc['l', 'Yes']

p_x_y

,No,Yes
Sunny,0.6,0.222222
Overcast,0.0,0.444444
Rain,0.4,0.333333


In [18]:
print(p_x_y.columns)
print(p_x_y)

Index(['No', 'Yes'], dtype='str')
           No       Yes
Sunny     0.6  0.222222
Overcast  0.0  0.444444
Rain      0.4  0.333333


Muestreo

In [19]:
sampled_values = np.random.choice(xvalues, size=10, p=p_x_y.loc[:,'Yes'].values)

sampled_values

array(['Overcast', 'Overcast', 'Sunny', 'Overcast', 'Overcast', 'Rain',
       'Overcast', 'Overcast', 'Overcast', 'Rain'], dtype=object)

## Aproxime $p(y,o,h,w,t)$

Aproxime la proabilidad conjunta del Tennis (y), Outlook (o), Humidity (h), Wind (w), Temp (t).

Recuerde que $p(y,o,h,w,t)$ = $p(y)$.$p(o|y)$.$p(h|y,o)$.$p(w|y,o,h)$.$p(t|y,o,h,w)$

Calcule P(y)

In [20]:
df

,Outlook,Temp,Humidity,Wind,Tennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [21]:
#Outlook

xvalues

<StringArray>
['Sunny', 'Overcast', 'Rain']
Length: 3, dtype: str

In [22]:
#Tennis

yvalues

<StringArray>
['No', 'Yes']
Length: 2, dtype: str

In [23]:
# P(Y)
p_y = pd.Series(0.0, index=yvalues)

for y in yvalues:
    p_y.loc[y] = len(df.loc[df['Tennis'] == y]) / N

p_y

No     0.357143
Yes    0.642857
dtype: float64

Calcule P(o|y)

In [24]:
# P(O|Y)
p_o_y = pd.DataFrame(0.0, columns=yvalues, index=xvalues)

for o in xvalues:
    for y in yvalues:
        count_oy = len(df.loc[(df['Outlook'] == o) & (df['Tennis'] == y)])
        count_y = len(df.loc[df['Tennis'] == y])
        p_o_y.loc[o, y] = count_oy / count_y

p_o_y

,No,Yes
Sunny,0.6,0.222222
Overcast,0.0,0.444444
Rain,0.4,0.333333


Calcule P(h|y,o)

Recomendamos usar la función *crosstab* de pandas. En este link pueden encontrar un ejemplo de su uso: https://www.geeksforgeeks.org/pandas-crosstab-function-in-python/

In [25]:
# Calcule la tabla de frecuencia
p_h_yo = pd.crosstab([df['Tennis'], df['Outlook']], df['Humidity'])

# Aseguramos que existan todas las combinaciones posibles de (Tennis, Outlook),
# incluso las que no tienen observaciones en el dataset (ej: Overcast-No)
full_index = pd.MultiIndex.from_product([yvalues, xvalues], names=['Tennis', 'Outlook'])
p_h_yo = p_h_yo.reindex(full_index)

# Luego se divide cada fila por la suma de sus elementos, puede usar las funciones div y sum de pandas
p_h_yo = p_h_yo.div(p_h_yo.sum(axis=1), axis=0)

# No se oliden de llenar los valores NaN!!!
p_h_yo = p_h_yo.fillna(0)

# Cambiamos los nombres de los indices (si es que lo precisa)

p_h_yo

Humidity             High    Normal
Tennis Outlook                     
No     Sunny     1.000000  0.000000
       Overcast  0.000000  0.000000
       Rain      0.500000  0.500000
Yes    Sunny     0.000000  1.000000
       Overcast  0.500000  0.500000
       Rain      0.333333  0.666667

Calcule P(w|y,o,h)

In [26]:
p_w_yoh = pd.crosstab([df['Tennis'], df['Outlook'], df['Humidity']], df['Wind'])

hvalues = list(X['Humidity'].unique())

full_index = pd.MultiIndex.from_product([yvalues, xvalues, hvalues], names=['Tennis', 'Outlook', 'Humidity'])
p_w_yoh = p_w_yoh.reindex(full_index)

p_w_yoh = p_w_yoh.div(p_w_yoh.sum(axis=1), axis=0)
p_w_yoh = p_w_yoh.fillna(0)

p_w_yoh

Wind                        Strong      Weak
Tennis Outlook  Humidity                    
No     Sunny    High      0.333333  0.666667
                Normal    0.000000  0.000000
       Overcast High      0.000000  0.000000
                Normal    0.000000  0.000000
       Rain     High      1.000000  0.000000
                Normal    1.000000  0.000000
Yes    Sunny    High      0.000000  0.000000
                Normal    0.500000  0.500000
       Overcast High      0.500000  0.500000
                Normal    0.500000  0.500000
       Rain     High      0.000000  1.000000
                Normal    0.000000  1.000000

Calcule P(t|y,o,h,w)

In [36]:
p_t_yohw = pd.crosstab([df['Tennis'], df['Outlook'], df['Humidity'], df['Wind']], df['Temp'])
wvalues = list(X['Wind'].unique())

full_index = pd.MultiIndex.from_product([yvalues, xvalues, hvalues, wvalues], names=['Tennis', 'Outlook', 'Humidity', 'Wind'])
p_t_yohw = p_t_yohw.reindex(full_index)

p_t_yohw = p_t_yohw.div(p_t_yohw.sum(axis=1), axis=0)
p_t_yohw = p_t_yohw.fillna(0)

p_t_yohw

Temp                             Cool  Hot  Mild
Tennis Outlook  Humidity Wind                   
No     Sunny    High     Weak     0.0  0.5   0.5
                         Strong   0.0  1.0   0.0
                Normal   Weak     0.0  0.0   0.0
                         Strong   0.0  0.0   0.0
       Overcast High     Weak     0.0  0.0   0.0
                         Strong   0.0  0.0   0.0
                Normal   Weak     0.0  0.0   0.0
                         Strong   0.0  0.0   0.0
       Rain     High     Weak     0.0  0.0   0.0
                         Strong   0.0  0.0   1.0
                Normal   Weak     0.0  0.0   0.0
                         Strong   1.0  0.0   0.0
Yes    Sunny    High     Weak     0.0  0.0   0.0
                         Strong   0.0  0.0   0.0
                Normal   Weak     1.0  0.0   0.0
                         Strong   0.0  0.0   1.0
       Overcast High     Weak     0.0  1.0   0.0
                         Strong   0.0  0.0   1.0
                Normal   Weak     0.0  1.0   0.0
                         Strong   1.0  0.0   0.0
       Rain     High     Weak     0.0  0.0   1.0
                         Strong   0.0  0.0   0.0
                Normal   Weak     0.5  0.0   0.5
                         Strong   0.0  0.0   0.0

Calcule P(y,o,h,w,t)

In [32]:
# Definimos una función que nos calcula la probabilidad conjunta usando la regla del producto.
def calculate_prob(y,o,h,w,t):
    "Calculate the probability of occurrence of a row"
    p = p_y.loc[y]
    p *= p_o_y.loc[o, y]
    p *= p_h_yo.loc[(y, o), h]
    p *= p_w_yoh.loc[(y, o, h), w]
    p *= p_t_yohw.loc[(y, o, h, w), t]
    return p

In [33]:
prob = calculate_prob('Yes','Sunny','Normal','Weak','Cool')

print(f'P(Yes|Sunny,Normal,Weak,Cool) = {prob}')
print(f"Cantidad de observaciones: {prob*N}")

P(Yes|Sunny,Normal,Weak,Cool) = 0.07142857142857142
Cantidad de observaciones: 1.0


Definimos el muestreo de datos completos.

Primero generamos un *y* utilizando *p(y)* y luego seguimos con las probabilidades condicionales.

In [35]:
tvalues = list(X['Temp'].unique())
tvalues

['Hot', 'Mild', 'Cool']

In [43]:
def sample_from_y(y):
    "Generates a sample of weather conditions based on a specific y"
    o = np.random.choice(xvalues, p=p_o_y.loc[:, y].values)
    h = np.random.choice(hvalues, p=p_h_yo.loc[(y,o)].values)
    w = np.random.choice(wvalues, p=p_w_yoh.loc[(y,o,h)].values)

    t_probs = p_t_yohw.loc[(y, o, h, w)].values
    if t_probs.sum() == 0:
        t_probs = np.ones(len(tvalues)) / len(tvalues)

    t = np.random.choice(tvalues, p=t_probs)
    return [y, o, h, w, t]

def sample():
    "Generates a sample of weather conditions based on a random y"
    y = np.random.choice(yvalues, p=p_y.values)
    return sample_from_y(y)

In [44]:
samples = np.array([sample() for _ in range(len(df))])
new_df = pd.DataFrame(samples, columns=['Tennis', 'Outlook', 'Humidity', 'Wind', 'Temp'])
new_df

,Tennis,Outlook,Humidity,Wind,Temp
0,Yes,Overcast,Normal,Weak,Mild
1,No,Sunny,High,Weak,Mild
2,Yes,Overcast,High,Strong,Cool
3,No,Rain,Normal,Weak,Cool
4,Yes,Sunny,Normal,Weak,Hot
5,Yes,Rain,Normal,Strong,Hot
6,Yes,Rain,Normal,Strong,Cool
7,Yes,Overcast,High,Strong,Cool
8,Yes,Overcast,High,Weak,Mild
9,No,Sunny,High,Strong,Mild
